# Insurance Claims Fraud Analysis: Exploration and Cleaning

**Goal:** find out which auto insurance claims are most likely to be fraudulent.

**Data:** 15,420 vehicle insurance claims (1994-1996), 33 columns. About 6% are labeled fraud.
**Limitation:** the label records fraud that was *detected*, so undetected fraud sits in the "not fraud" group.

## 1. Data quality checks

In [1]:
# Load raw data and run basic quality checks

import pandas as pd

pd.set_option("display.max_columns", None)

df = pd.read_csv("../Data/fraud_oracle.csv")

print("Shape:", df.shape)
print("\nMissing values:", df.isnull().sum().sum())
print("Duplicate rows:", df.duplicated().sum())
df.head()

Shape: (15420, 33)

Missing values: 0
Duplicate rows: 0


,Month,WeekOfMonth,DayOfWeek,Make,AccidentArea,DayOfWeekClaimed,MonthClaimed,WeekOfMonthClaimed,Sex,MaritalStatus,Age,Fault,PolicyType,VehicleCategory,VehiclePrice,FraudFound_P,PolicyNumber,RepNumber,Deductible,DriverRating,Days_Policy_Accident,Days_Policy_Claim,PastNumberOfClaims,AgeOfVehicle,AgeOfPolicyHolder,PoliceReportFiled,WitnessPresent,AgentType,NumberOfSuppliments,AddressChange_Claim,NumberOfCars,Year,BasePolicy
0,Dec,5,Wednesday,Honda,Urban,Tuesday,Jan,1,Female,Single,21,Policy Holder,Sport - Liability,Sport,more than 69000,0,1,12,300,1,more than 30,more than 30,none,3 years,26 to 30,No,No,External,none,1 year,3 to 4,1994,Liability
1,Jan,3,Wednesday,Honda,Urban,Monday,Jan,4,Male,Single,34,Policy Holder,Sport - Collision,Sport,more than 69000,0,2,15,400,4,more than 30,more than 30,none,6 years,31 to 35,Yes,No,External,none,no change,1 vehicle,1994,Collision
2,Oct,5,Friday,Honda,Urban,Thursday,Nov,2,Male,Married,47,Policy Holder,Sport - Collision,Sport,more than 69000,0,3,7,400,3,more than 30,more than 30,1,7 years,41 to 50,No,No,External,none,no change,1 vehicle,1994,Collision
3,Jun,2,Saturday,Toyota,Rural,Friday,Jul,1,Male,Married,65,Third Party,Sedan - Liability,Sport,20000 to 29000,0,4,4,400,2,more than 30,more than 30,1,more than 7,51 to 65,Yes,No,External,more than 5,no change,1 vehicle,1994,Liability
4,Jan,5,Monday,Honda,Urban,Tuesday,Feb,2,Female,Single,27,Third Party,Sport - Collision,Sport,more than 69000,0,5,3,400,1,more than 30,more than 30,none,5 years,31 to 35,No,No,External,none,no change,1 vehicle,1994,Collision


In [2]:
# Column types and non-null counts
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 15420 entries, 0 to 15419
Data columns (total 33 columns):
 #   Column                Non-Null Count  Dtype
---  ------                --------------  -----
 0   Month                 15420 non-null  str  
 1   WeekOfMonth           15420 non-null  int64
 2   DayOfWeek             15420 non-null  str  
 3   Make                  15420 non-null  str  
 4   AccidentArea          15420 non-null  str  
 5   DayOfWeekClaimed      15420 non-null  str  
 6   MonthClaimed          15420 non-null  str  
 7   WeekOfMonthClaimed    15420 non-null  int64
 8   Sex                   15420 non-null  str  
 9   MaritalStatus         15420 non-null  str  
 10  Age                   15420 non-null  int64
 11  Fault                 15420 non-null  str  
 12  PolicyType            15420 non-null  str  
 13  VehicleCategory       15420 non-null  str  
 14  VehiclePrice          15420 non-null  str  
 15  FraudFound_P          15420 non-null  int64
 16  PolicyNumber   

In [3]:
# Check text columns for odd values; check duplicates and ID column

for col in df.select_dtypes(include=["object", "string"]).columns:
    print(col, df[col].nunique(), "->", sorted(df[col].unique())[:12])

print("Duplicate rows:", df.duplicated().sum())
print("PolicyNumber unique:", df["PolicyNumber"].nunique(), "of", len(df))

Month 12 -> ['Apr', 'Aug', 'Dec', 'Feb', 'Jan', 'Jul', 'Jun', 'Mar', 'May', 'Nov', 'Oct', 'Sep']
DayOfWeek 7 -> ['Friday', 'Monday', 'Saturday', 'Sunday', 'Thursday', 'Tuesday', 'Wednesday']
Make 19 -> ['Accura', 'BMW', 'Chevrolet', 'Dodge', 'Ferrari', 'Ford', 'Honda', 'Jaguar', 'Lexus', 'Mazda', 'Mecedes', 'Mercury']
AccidentArea 2 -> ['Rural', 'Urban']
DayOfWeekClaimed 8 -> ['0', 'Friday', 'Monday', 'Saturday', 'Sunday', 'Thursday', 'Tuesday', 'Wednesday']
MonthClaimed 13 -> ['0', 'Apr', 'Aug', 'Dec', 'Feb', 'Jan', 'Jul', 'Jun', 'Mar', 'May', 'Nov', 'Oct']
Sex 2 -> ['Female', 'Male']
MaritalStatus 4 -> ['Divorced', 'Married', 'Single', 'Widow']
Fault 2 -> ['Policy Holder', 'Third Party']
PolicyType 9 -> ['Sedan - All Perils', 'Sedan - Collision', 'Sedan - Liability', 'Sport - All Perils', 'Sport - Collision', 'Sport - Liability', 'Utility - All Perils', 'Utility - Collision', 'Utility - Liability']
VehicleCategory 3 -> ['Sedan', 'Sport', 'Utility']
VehiclePrice 6 -> ['20000 to 29000'

In [4]:
# Inspect Make typos and the rows with '0' claim dates

print(sorted(df["Make"].unique()))

bad = df[(df["MonthClaimed"] == "0") | (df["DayOfWeekClaimed"] == "0")]
print("Rows with a '0' claim date:", len(bad))
print("Fraud cases among them:", bad["FraudFound_P"].sum())
print(bad[["Month", "MonthClaimed", "DayOfWeekClaimed", "Days_Policy_Claim", "FraudFound_P"]])

['Accura', 'BMW', 'Chevrolet', 'Dodge', 'Ferrari', 'Ford', 'Honda', 'Jaguar', 'Lexus', 'Mazda', 'Mecedes', 'Mercury', 'Nisson', 'Pontiac', 'Porche', 'Saab', 'Saturn', 'Toyota', 'VW']
Rows with a '0' claim date: 1
Fraud cases among them: 0
     Month MonthClaimed DayOfWeekClaimed Days_Policy_Claim  FraudFound_P
1516   Jul            0                0              none             0


## 2. Fraud rate by segment

In [5]:
# Target variable: how rare is fraud?

print(df["FraudFound_P"].value_counts())
print(df["FraudFound_P"].value_counts(normalize=True).round(4) * 100)

FraudFound_P
0    14497
1      923
Name: count, dtype: int64
FraudFound_P
0    94.01
1     5.99
Name: proportion, dtype: float64


In [6]:
# Helper: claims, fraud cases and fraud rate (%) by column

def fraud_rate(col):
    out = (df.groupby(col)["FraudFound_P"]
             .agg(claims="count", fraud_cases="sum", fraud_rate="mean")
             .sort_values("fraud_rate", ascending=False))
    out["fraud_rate"] = (out["fraud_rate"] * 100).round(2)
    return out

for col in ["PolicyType", "VehicleCategory", "BasePolicy", "Fault", "AgentType"]:
    print(f"--- {col} ---")
    print(fraud_rate(col))
    print()

--- PolicyType ---
                      claims  fraud_cases  fraud_rate
PolicyType                                           
Sport - Collision        348           48       13.79
Utility - All Perils     340           41       12.06
Sedan - All Perils      4087          411       10.06
Utility - Collision       30            3       10.00
Sedan - Collision       5584          384        6.88
Sedan - Liability       4987           36        0.72
Sport - All Perils        22            0        0.00
Sport - Liability          1            0        0.00
Utility - Liability       21            0        0.00

--- VehicleCategory ---
                 claims  fraud_cases  fraud_rate
VehicleCategory                                 
Utility             391           44       11.25
Sedan              9671          795        8.22
Sport              5358           84        1.57

--- BasePolicy ---
            claims  fraud_cases  fraud_rate
BasePolicy                                 
All Peril

In [7]:
# Fraud rate by coverage type and fault

(df.groupby(["BasePolicy", "Fault"])["FraudFound_P"]
   .agg(claims="count", fraud_cases="sum", fraud_rate="mean")
   .assign(fraud_rate=lambda d: (d["fraud_rate"] * 100).round(2)))

claims  fraud_cases  fraud_rate
BasePolicy Fault                                         
All Perils Policy Holder    2797          436       15.59
           Third Party      1652           16        0.97
Collision  Policy Holder    4151          415       10.00
           Third Party      1811           20        1.10
Liability  Policy Holder    4282           35        0.82
           Third Party       727            1        0.14

In [8]:
# findings from above exploration (risk segment)

risk = df["BasePolicy"].isin(["All Perils", "Collision"]) & (df["Fault"] == "Policy Holder")
print("Share of claims:", round(risk.mean() * 100, 1), "%")
print("Fraud rate in segment:", round(df.loc[risk, "FraudFound_P"].mean() * 100, 2), "%")
print("Share of all fraud captured:", round(df.loc[risk, "FraudFound_P"].sum() / df["FraudFound_P"].sum() * 100, 1), "%")

Share of claims: 45.1 %
Fraud rate in segment: 12.25 %
Share of all fraud captured: 92.2 %


“Fraud is concentrated in at-fault claims on All Perils and Collision policies: 12.25% fraud rate, capturing 92.2% of all fraud.”

In [9]:
# Fraud rate by past claims, days to claim, vehicle price, policyholder age

for col in ["PastNumberOfClaims", "Days_Policy_Claim", "VehiclePrice", "AgeOfPolicyHolder"]:
    print(fraud_rate(col))
    print()

                    claims  fraud_cases  fraud_rate
PastNumberOfClaims                                 
none                  4352          339        7.79
1                     3573          222        6.21
2 to 4                5485          294        5.36
more than 4           2010           68        3.38

                   claims  fraud_cases  fraud_rate
Days_Policy_Claim                                 
8 to 15                21            3       14.29
15 to 30               56            6       10.71
more than 30        15342          914        5.96
none                    1            0        0.00

                 claims  fraud_cases  fraud_rate
VehiclePrice                                    
less than 20000    1096          103        9.40
more than 69000    2164          189        8.73
40000 to 59000      461           31        6.72
20000 to 29000     8079          421        5.21
30000 to 39000     3533          175        4.95
60000 to 69000       87            4 

In [10]:
# Summary stats; Age minimum of 0 is suspicious

df[["Age", "Deductible", "DriverRating", "Year"]].describe()

,Age,Deductible,DriverRating,Year
count,15420.000000,15420.000000,15420.000000,15420.000000
mean,39.855707,407.704280,2.487808,1994.866472
std,13.492377,43.950998,1.119453,0.803313
min,0.000000,300.000000,1.000000,1994.000000
25%,31.000000,400.000000,1.000000,1994.000000
50%,38.000000,400.000000,2.000000,1995.000000
75%,48.000000,400.000000,3.000000,1996.000000
max,80.000000,700.000000,4.000000,1996.000000


In [11]:
# Investigate Age = 0: which age band are these rows in?

print("Rows with Age = 0:", (df["Age"] == 0).sum())
print(df.loc[df["Age"] == 0, "AgeOfPolicyHolder"].value_counts())
print("Fraud rate where Age = 0:", round(df.loc[df["Age"] == 0, "FraudFound_P"].mean() * 100, 2), "%")

Rows with Age = 0: 320
AgeOfPolicyHolder
16 to 17    320
Name: count, dtype: int64
Fraud rate where Age = 0: 9.69 %


“Fixes: 320 Age = 0 placeholders flagged and set to missing, 1 row with a placeholder claim date marked Unknown, 4 misspelled car makes corrected. No duplicates, no missing values.”

## 3. Data cleaning

In [12]:
# DATA CLEANING: fix placeholders and typos, save cleaned file

import numpy as np

# Age: 0 is a placeholder for the 16-17 band
df["AgeMissing"] = ((df["Age"] == 0) | df["Age"].isna()).astype(int)
df["Age"] = df["Age"].replace(0, np.nan)

# Claim date placeholders (1 row)
df["MonthClaimed"] = df["MonthClaimed"].replace("0", "Unknown")
df["DayOfWeekClaimed"] = df["DayOfWeekClaimed"].replace("0", "Unknown")

# Typos in Make
df["Make"] = df["Make"].replace({
    "Accura": "Acura",
    "Mecedes": "Mercedes",
    "Nisson": "Nissan",
    "Porche": "Porsche",
})

# Quick checks
print("Age placeholders:", df["AgeMissing"].sum())                 # 320
print("Unknown claim dates:", (df["MonthClaimed"] == "Unknown").sum())   # 1
print(sorted(df["Make"].unique()))

df.to_csv("../Data/claims_clean.csv", index=False)

Age placeholders: 320
Unknown claim dates: 1
['Acura', 'BMW', 'Chevrolet', 'Dodge', 'Ferrari', 'Ford', 'Honda', 'Jaguar', 'Lexus', 'Mazda', 'Mercedes', 'Mercury', 'Nissan', 'Pontiac', 'Porsche', 'Saab', 'Saturn', 'Toyota', 'VW']


In [13]:
# Export a copy without Age for the MySQL import (the wizard skips blank numeric values)

sql_df = df.drop(columns=["Age"])
sql_df.to_csv("../Data/claims_mysql.csv", index=False)
print(sql_df.shape)   # should be (15420, 33)

(15420, 33)
